# Tools: Giving Your Agent Superpowers

Language models are great at understanding and generating text, but they have real limitations:

- They can't do precise math (try asking one to multiply 3.1415 by 2.7182!)
- They can't look up today's weather or news
- They can't interact with databases, APIs, or files

**Tools** solve this problem. A tool is just a Python function that you give to the agent. When the agent needs to do something it can't do on its own, it **calls the tool** and uses the result.

By the end of this notebook, you will be able to:
- Create tools using the `@tool` decorator
- Give tools to an agent and watch it decide when to use them
- Understand the message flow: Human → AI → Tool → AI
- Give an agent **multiple tools** and let it pick the right one
- Combine tools with **system messages** and **structured output** to build a complete mini-agent

## Setup

Make sure you have a `.env` file in your project directory with the following variables:

```
OPENAI_API_KEY=your-api-key-here
OPENAI_ENDPOINT=your-endpoint-here
```

Run the cells below to load them and verify everything is working.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

# Verify that the required environment variables are set
assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY is not set! Check your .env file."
assert os.environ.get("OPENAI_ENDPOINT"), "OPENAI_ENDPOINT is not set! Check your .env file."

print("Environment variables loaded successfully!")

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(base_url=os.environ["OPENAI_ENDPOINT"], model="gpt-5.4-mini")

# Quick test - make sure the model is working
response = llm.invoke("Say hello in one sentence!")
print(response.content)

---

## Part 1: Your First Tool

A **tool** is just a Python function with two special things:

1. The `@tool` decorator on top
2. A **docstring** that describes what the function does (this is what the LLM reads to decide when to use it!)

Let's create a simple tool that adds two numbers:

In [ ]:
from langchain_core.tools import tool


@tool
def add(a: int, b: int) -> int:
    """Add two numbers together and return the result."""
    return a + b

That's it! Three things to notice:

- `@tool` turns a regular function into a LangChain tool
- The **type hints** (`a: int, b: int`) tell the LLM what kind of inputs are expected
- The **docstring** (`"""Add two numbers..."""`) is the most important part — it's what the LLM reads to decide *when* to call this tool

Now let's give this tool to an agent:

In [ ]:
from langchain.agents import create_agent

# Create an agent that can use our add tool
math_agent = create_agent(
    model=llm,
    tools=[add],
    system_prompt="You are a helpful math assistant. Use your tools when asked to do math.",
)

result = math_agent.invoke({"messages": "What is 15 + 27?"})
print(result["messages"][-1].content)

The agent didn't just *guess* the answer — it actually **called our `add` function** to compute it!

### Seeing the full message flow

Let's look at *all* the messages to understand what happened behind the scenes:

In [ ]:
result = math_agent.invoke({"messages": "What is 100 + 250?"})

for msg in result["messages"]:
    msg.pretty_print()

Here's what happened, step by step:

| Step | Message Type | What happened |
|------|-------------|---------------|
| 1 | **HumanMessage** | You asked "What is 100 + 250?" |
| 2 | **AIMessage** | The model decided to call the `add` tool with `a=100, b=250` |
| 3 | **ToolMessage** | The tool ran and returned `350` |
| 4 | **AIMessage** | The model used the tool result to give you the final answer |

This is the **agent loop**: the LLM *thinks*, *acts* (calls a tool), *observes* (reads the result), and then *responds*.

### What if no tool is needed?

The agent is smart enough to **not** use a tool when it doesn't need one:

In [ ]:
result = math_agent.invoke({"messages": "What is the capital of France?"})

for msg in result["messages"]:
    msg.pretty_print()

Notice: no `ToolMessage` this time! The agent answered directly because the question didn't require any calculation.

---

## Part 2: The Docstring Matters!

The LLM reads the **tool name**, **parameter names/types**, and most importantly the **docstring** to decide which tool to use and when.

Let's create a tool with a more detailed description:

In [ ]:
@tool
def get_weather(city: str) -> str:
    """Get the current weather for a city. Returns temperature and conditions."""
    # In a real app, this would call a weather API.
    # For now, we'll return fake data.
    weather_data = {
        "copenhagen": "15°C, partly cloudy",
        "new york": "22°C, sunny",
        "tokyo": "28°C, humid",
        "london": "12°C, rainy",
    }
    return weather_data.get(city.lower(), f"Sorry, I don't have weather data for {city}.")


weather_agent = create_agent(
    model=llm,
    tools=[get_weather],
    system_prompt="You are a weather assistant. Use the get_weather tool to look up weather.",
)

result = weather_agent.invoke({"messages": "What's the weather like in Copenhagen?"})
print(result["messages"][-1].content)

In [ ]:
# Let's see what happens with a city we don't have data for
result = weather_agent.invoke({"messages": "How's the weather in Paris?"})
print(result["messages"][-1].content)

The tool returned "Sorry, I don't have weather data for Paris" — and the agent relayed that to the user in a helpful way. The agent handles tool errors gracefully!

---

## Part 3: Multiple Tools

The real power of agents comes when you give them **multiple tools**. The LLM reads all the tool descriptions and picks the right one based on your question.

Let's give an agent both math and weather tools:

In [ ]:
@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers together and return the result."""
    return a * b


# Give the agent three tools: add, multiply, and get_weather
multi_agent = create_agent(
    model=llm,
    tools=[add, multiply, get_weather],
    system_prompt="You are a helpful assistant with access to math and weather tools.",
)

In [ ]:
# Ask a math question - should use 'add'
result = multi_agent.invoke({"messages": "What is 5 + 3?"})
for msg in result["messages"]:
    msg.pretty_print()

In [ ]:
# Ask a different math question - should use 'multiply'
result = multi_agent.invoke({"messages": "What is 7 times 8?"})
for msg in result["messages"]:
    msg.pretty_print()

In [ ]:
# Ask about weather - should use 'get_weather'
result = multi_agent.invoke({"messages": "What's the weather in Tokyo?"})
for msg in result["messages"]:
    msg.pretty_print()

The agent picked the right tool each time! It used `add` for addition, `multiply` for multiplication, and `get_weather` for weather. This is why good docstrings matter — they help the LLM make the right choice.

### Multiple tool calls in one question

Sometimes the agent needs to call **more than one tool** to answer a single question:

In [ ]:
result = multi_agent.invoke(
    {"messages": "What is 10 + 20, and also what is 3 times 7?"}
)
for msg in result["messages"]:
    msg.pretty_print()

The agent called both `add` and `multiply` to answer one question!

---

## Part 4: Tools with More Interesting Logic

Tools can do anything a Python function can do. Let's create some more useful ones:

In [ ]:
@tool
def count_words(text: str) -> int:
    """Count the number of words in a given text."""
    return len(text.split())


@tool
def reverse_text(text: str) -> str:
    """Reverse the given text (flip it backwards)."""
    return text[::-1]


@tool
def to_uppercase(text: str) -> str:
    """Convert the given text to all uppercase letters."""
    return text.upper()


text_agent = create_agent(
    model=llm,
    tools=[count_words, reverse_text, to_uppercase],
    system_prompt="You are a text processing assistant. Use your tools to manipulate text when asked.",
)

In [ ]:
result = text_agent.invoke({"messages": "How many words are in 'The quick brown fox jumps over the lazy dog'?"})
print(result["messages"][-1].content)

In [ ]:
result = text_agent.invoke({"messages": "Reverse the word 'hello'"})
print(result["messages"][-1].content)

In [ ]:
result = text_agent.invoke({"messages": "Make 'whisper' uppercase"})
print(result["messages"][-1].content)

---

## Part 5: Putting It All Together

Now let's combine everything we've learned across lectures:

- **System message** — to give the agent a personality and instructions
- **Tools** — to let the agent take actions
- **Structured output** — to get the result in a predictable format

### Example: A Travel Assistant

Let's build a travel assistant that can look up weather and give you a structured travel recommendation.

In [ ]:
from pydantic import BaseModel


# Step 1: Define the structured output format
class TravelRecommendation(BaseModel):
    city: str
    weather: str
    recommendation: str
    pack_items: list[str]

In [ ]:
# Step 2: Create the tool
@tool
def lookup_weather(city: str) -> str:
    """Look up the current weather for a city. Returns temperature and conditions."""
    weather_data = {
        "copenhagen": "15°C, partly cloudy with chance of rain",
        "barcelona": "30°C, sunny and hot",
        "tokyo": "28°C, humid with thunderstorms expected",
        "reykjavik": "5°C, windy and cold",
        "new york": "22°C, clear skies",
    }
    return weather_data.get(city.lower(), f"Weather data not available for {city}.")

In [ ]:
# Step 3: Create the agent with system prompt, tools, AND structured output
travel_agent = create_agent(
    model=llm,
    tools=[lookup_weather],
    system_prompt=(
        "You are a friendly travel assistant. "
        "When someone asks about traveling to a city, use the lookup_weather tool "
        "to check the weather, then give them a recommendation and packing list."
    ),
    response_format=TravelRecommendation,
)

In [ ]:
# Step 4: Use it!
result = travel_agent.invoke({"messages": "I'm thinking of visiting Barcelona this weekend."})

rec = result["structured_response"]
print(f"City: {rec.city}")
print(f"Weather: {rec.weather}")
print(f"Recommendation: {rec.recommendation}")
print(f"Pack these items: {rec.pack_items}")

In [ ]:
# Try another city
result = travel_agent.invoke({"messages": "Should I go to Reykjavik next week?"})

rec = result["structured_response"]
print(f"City: {rec.city}")
print(f"Weather: {rec.weather}")
print(f"Recommendation: {rec.recommendation}")
print(f"Pack these items: {rec.pack_items}")

Look at what just happened:

1. The **system message** told the agent to act as a travel assistant
2. The agent used the **tool** to look up the weather
3. The **structured output** ensured we always get back a `city`, `weather`, `recommendation`, and `pack_items`

This is the foundation of building real AI agents: give the LLM a role, give it tools to interact with the world, and define the output format you need.

---

## Part 6: Exercises

Now it's your turn! Complete the exercises below.

### Exercise 1: Create a Simple Tool

Create a tool called `subtract` that takes two numbers and returns the difference. Then create an agent that uses it and test with a question.

In [ ]:
# TODO: Create a subtract tool using @tool
# @tool
# def subtract(a: int, b: int) -> int:
#     """..."""
#     ...

# TODO: Create an agent with the subtract tool
# agent = create_agent(model=llm, tools=[subtract], system_prompt="...")

# TODO: Test it
# result = agent.invoke({"messages": "What is 100 minus 37?"})
# print(result["messages"][-1].content)

### Exercise 2: Multi-Tool Agent

Create an agent with at least **3 tools** of your choice. They can be anything — math, text, lookup, etc. Test it with different questions to make sure it picks the right tool each time.

**Hint:** Think about what makes a good docstring. The LLM uses it to decide which tool to call!

In [ ]:
# TODO: Create at least 3 tools

# TODO: Create an agent with all your tools

# TODO: Test with different questions - one per tool


### Exercise 3: Structured Output + Tools

Build a **food ordering assistant**. It should:

1. Have a tool called `get_menu` that takes a `restaurant` name (str) and returns a menu (just return a hardcoded string with a few items and prices)
2. Have a tool called `calculate_total` that takes a `prices` list (list of floats) and returns the total
3. Return a structured output using Pydantic with these fields:
   - `restaurant` (str)
   - `items_ordered` (list[str])
   - `total_price` (float)
   - `estimated_wait` (str)

Give it a system prompt that makes it a friendly food ordering assistant.

Test it by asking something like: *"I'd like to order a burger and fries from Joe's Diner"*

In [ ]:
# TODO: Define the Pydantic schema for the order
# class FoodOrder(BaseModel):
#     restaurant: str
#     items_ordered: list[str]
#     total_price: float
#     estimated_wait: str

# TODO: Create the get_menu tool
# @tool
# def get_menu(restaurant: str) -> str:
#     """..."""
#     ...

# TODO: Create the calculate_total tool
# @tool
# def calculate_total(prices: list[float]) -> float:
#     """..."""
#     ...

# TODO: Create the agent with system_prompt, tools, AND response_format
# food_agent = create_agent(
#     model=llm,
#     tools=[get_menu, calculate_total],
#     system_prompt="...",
#     response_format=FoodOrder,
# )

# TODO: Test it and print the structured response
# result = food_agent.invoke({"messages": "I'd like a burger and fries from Joe's Diner"})
# order = result["structured_response"]
# print(f"Restaurant: {order.restaurant}")
# print(f"Items: {order.items_ordered}")
# print(f"Total: ${order.total_price}")
# print(f"Wait time: {order.estimated_wait}")

### Bonus Exercise: Build Your Own Mini-Agent

Design and build your own agent from scratch! Pick a theme (e.g., fitness tracker, movie recommender, study planner, quiz master) and:

1. Create **2+ tools** relevant to your theme
2. Write a **system prompt** that gives the agent a personality
3. Define a **Pydantic schema** for the output
4. Test it with a few different inputs

Be creative!

In [ ]:
# Your creative agent here!
